# Week 3 — Assumptions, Experimental Design, Small N, and Large p

This week focuses on a question that comes **before** choosing a statistical test:

> **What is the structure of the experiment and data, and what does that structure allow us to infer?**

By the end of class, you should be able to:

1. Distinguish assumptions that come from **study design** from assumptions that can be checked with model diagnostics.
2. Explain how **effect size, variability, and sample size** influence statistical power.
3. Explain why high-dimensional neuroscience data create a **multiple-testing** problem.


## 0. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.multitest import multipletests

# Reproducible random-number generator
rng = np.random.default_rng(3061)


## 1. Start with the experimental unit

Suppose we measure firing rates from **20 neurons in each mouse** after assigning mice to either control or drug treatment.

A spreadsheet might contain 240 rows:

- 6 control mice × 20 neurons = 120 rows
- 6 drug mice × 20 neurons = 120 rows

But are there really 120 independent biological replicates per condition?

The **experimental unit** is the smallest unit that could have been independently assigned to a treatment. Here, treatment was assigned to the **mouse**, so the mouse—not each neuron—is the experimental unit for the treatment effect.


In [ ]:
# Synthetic nested neuroscience dataset:
# 6 mice per condition, 20 neurons per mouse.
rows = []

for condition, population_mean in [('control', 10.0), ('drug', 10.8)]:
    for mouse_num in range(1, 7):
        # Mice differ from one another biologically.
        mouse_mean = rng.normal(loc=population_mean, scale=1.1)

        # Neurons from the same mouse cluster around that mouse's mean.
        neuron_values = rng.normal(loc=mouse_mean, scale=0.9, size=20)

        for neuron_num, value in enumerate(neuron_values, start=1):
            rows.append({
                'condition': condition,
                'mouse': f'{condition}_{mouse_num}',
                'neuron': neuron_num,
                'firing_rate_hz': value
            })

nested = pd.DataFrame(rows)
nested.head()


In [ ]:
print('Rows in dataframe:', len(nested))
print('Unique mice:', nested['mouse'].nunique())
print('\nRows per condition:')
print(nested.groupby('condition').size())
print('\nMice per condition:')
print(nested.groupby('condition')['mouse'].nunique())


### 1.1 A tempting but incorrect analysis

If we treat every neuron as an independent replicate, the nominal sample size becomes 120 per group. That ignores the fact that neurons from the same mouse share genetics, treatment history, tissue preparation, and other sources of variation.

This is **pseudoreplication**: counting subsamples as if they were independent experimental units.


In [ ]:
control_cells = nested.loc[nested['condition'] == 'control', 'firing_rate_hz']
drug_cells = nested.loc[nested['condition'] == 'drug', 'firing_rate_hz']

naive_test = stats.ttest_ind(control_cells, drug_cells, equal_var=False)
print('Naive neuron-level comparison')
print('Mean difference:', drug_cells.mean() - control_cells.mean())
print('p-value:', naive_test.pvalue)


### 1.2 Respect the unit of treatment assignment

One simple approach is to summarize neurons within each mouse and compare mouse-level values. More advanced models can retain neuron-level information while explicitly modeling the nesting (for example, mixed-effects models), but the dependence cannot simply be ignored.


In [ ]:
mouse_means = (
    nested.groupby(['condition', 'mouse'], observed=True)['firing_rate_hz']
    .mean()
    .reset_index()
)

mouse_means


In [ ]:
control_mice = mouse_means.loc[mouse_means['condition'] == 'control', 'firing_rate_hz']
drug_mice = mouse_means.loc[mouse_means['condition'] == 'drug', 'firing_rate_hz']

mouse_level_test = stats.ttest_ind(control_mice, drug_mice, equal_var=False)
print('Mouse-level comparison')
print('Mean difference:', drug_mice.mean() - control_mice.mean())
print('p-value:', mouse_level_test.pvalue)


In [ ]:
# Visualize both the mouse-level values and the within-mouse neuronal observations.
fig, ax = plt.subplots(figsize=(8, 4.5))

x_positions = {'control': 0, 'drug': 1}
for condition in ['control', 'drug']:
    sub = nested[nested['condition'] == condition]
    for mouse, mouse_df in sub.groupby('mouse'):
        x = x_positions[condition] + rng.normal(0, 0.035, size=len(mouse_df))
        ax.scatter(x, mouse_df['firing_rate_hz'], alpha=0.18, s=16)

    mm = mouse_means[mouse_means['condition'] == condition]['firing_rate_hz']
    x = x_positions[condition] + rng.normal(0, 0.07, size=len(mm))
    ax.scatter(x, mm, s=70, linewidth=0.8, label=f'{condition}: mouse means')

ax.set_xticks([0, 1], ['Control', 'Drug'])
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Many neurons, but only 6 independent mice per condition')
ax.legend()
plt.show()


## 2. Assumptions: what can we check, and what must come from the design?

Statistical assumptions are not a single “normality checklist.” Different assumptions enter at different levels.

| Assumption / issue | Can a plot or test diagnose it? | Where does the evidence come from? |
|---|---:|---|
| Independence of experimental units | Usually **no** | Study design / sampling scheme |
| Correct pairing or repeated-measure structure | Usually **no** | Study design |
| Approximate residual normality | Partly | Residual histogram / Q–Q plot |
| Similar residual variance across fitted values or groups | Partly | Residual plots / group spreads |
| Linearity for continuous predictors | Partly | Scatterplots / residual plots |
| Influential outliers | Partly | Raw data + model diagnostics |

A formal assumption test can itself have low power at small sample sizes and excessive sensitivity at very large sample sizes. Visual inspection plus scientific context is often more informative than using one diagnostic p-value.


### 2.1 Fit a simple model at the correct experimental level

A two-group comparison can be written as a regression model. Here we fit the model to the **mouse means**, because mice are the independent units for the treatment assignment.


In [ ]:
model = smf.ols('firing_rate_hz ~ C(condition)', data=mouse_means).fit()
model.summary()


In [ ]:
residuals = model.resid
fitted = model.fittedvalues

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fitted, residuals, s=60)
ax.axhline(0, linewidth=1)
ax.set_xlabel('Fitted value')
ax.set_ylabel('Residual')
ax.set_title('Residuals vs fitted values')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Q–Q plot of model residuals')
plt.show()


### Read diagnostics cautiously

With only 12 independent mice, diagnostic plots will look sparse. That is itself an important lesson: **small N limits our ability to diagnose assumptions precisely**.

Ask:

- Is there an obvious systematic pattern?
- Is one observation dominating the result?
- Are group variances radically different?
- Is the statistical model aligned with the experimental design?
- Would a robust, transformed, nonparametric, or hierarchical analysis better answer the scientific question?

Do not interpret “the Shapiro–Wilk test was not significant” as proof that data are normal.


## 3. Small N: effect size, uncertainty, and power

A non-significant result can arise because:

- the true effect is near zero,
- the effect exists but is smaller than expected,
- biological variability is large,
- the experiment has too few independent units,
- measurement noise is high,
- the model does not match the design.

**Power** is the probability that a study will reject the null hypothesis when a specified nonzero effect is truly present, under the assumptions of the power calculation.

For a standardized two-group effect,

$$
 d = \frac{\mu_1 - \mu_2}{\sigma}
$$



In [ ]:
power_analysis = TTestIndPower()

sample_sizes = np.arange(4, 51)
effect_sizes = [0.5, 0.8, 1.0]

fig, ax = plt.subplots(figsize=(7, 4.5))
for d in effect_sizes:
    power = power_analysis.power(
        effect_size=d,
        nobs1=sample_sizes,
        alpha=0.05,
        ratio=1.0,
        alternative='two-sided'
    )
    ax.plot(sample_sizes, power, label=f'Cohen d = {d}')

ax.axhline(0.80, linestyle='--', linewidth=1)
ax.set_xlabel('Independent experimental units per group')
ax.set_ylabel('Power')
ax.set_ylim(0, 1.02)
ax.set_title('Power depends on effect size and independent sample size')
ax.legend()
plt.show()


In [ ]:
for d in [0.5, 0.8, 1.0]:
    n_needed = power_analysis.solve_power(
        effect_size=d,
        power=0.80,
        alpha=0.05,
        ratio=1.0,
        alternative='two-sided'
    )
    print(f'd = {d:0.1f}: about {np.ceil(n_needed):.0f} independent units per group for 80% power')


### Important caveats about power

- The assumed effect size should come from biological reasoning, prior evidence, or a **smallest effect size of interest**, not from choosing a convenient number after seeing the data.
- Adding repeated measurements within one animal is not the same as adding independent animals.
- A post-hoc power calculation based only on the observed p-value rarely adds useful information. Confidence intervals and effect estimates are usually more informative after the study is complete.
- Ethical and practical constraints matter. Power analysis is one input to experimental design, not the only input.


## 4. Large p: many variables create many chances for false positives

Examples:

- thousands of genes,
- hundreds of brain regions or connections,
- many time points or frequency bins,
- thousands of imaging voxels or extracted image features.

If we test each feature independently at $\alpha=0.05$, some features will appear “significant” by chance even when there is no true group difference.


In [ ]:
# Simulate a null experiment with 20 subjects and 1,000 features.
# There is deliberately NO true condition effect on any feature.
rng_large_p = np.random.default_rng(2026)

n_per_group = 10
n_features = 1000

control = rng_large_p.normal(0, 1, size=(n_per_group, n_features))
drug = rng_large_p.normal(0, 1, size=(n_per_group, n_features))

p_values = np.array([
    stats.ttest_ind(control[:, j], drug[:, j], equal_var=False).pvalue
    for j in range(n_features)
])

print('Features tested:', n_features)
print('p < 0.05:', (p_values < 0.05).sum())
print('Smallest p-value:', p_values.min())


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(p_values, bins=20)
ax.axvline(0.05, linestyle='--', linewidth=1)
ax.set_xlabel('Uncorrected p-value')
ax.set_ylabel('Number of features')
ax.set_title('Under the global null, small p-values still occur by chance')
plt.show()


### 4.1 A preview of multiple-comparison correction

Two common goals are:

- **Family-wise error rate (FWER):** strongly control the probability of making even one false positive in a family of tests.
- **False discovery rate (FDR):** control the expected proportion of false discoveries among the rejected hypotheses. 


In [ ]:
bonferroni_alpha = 0.05 / n_features
reject_bonf = p_values < bonferroni_alpha
reject_fdr, pvals_fdr, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

print('Bonferroni threshold:', bonferroni_alpha)
print('Bonferroni discoveries:', reject_bonf.sum())
print('Benjamini–Hochberg FDR discoveries:', reject_fdr.sum())


## 5. Practical decision framework

Before interpreting a p-value, work through these questions in order:

1. **What is the scientific question?**
2. **What is the experimental unit?**
3. **Which observations are truly independent?**
4. **Are there paired, repeated, nested, or clustered measurements?**
5. **What does the raw data look like?**
6. **What assumptions does the proposed model make?**
7. **Is the independent sample size adequate for the effect you care about?**
8. **How many hypotheses or features are being tested?**
9. **What effect estimate and uncertainty should be reported?**
10. **What limitations should remain explicit?**

> A sophisticated statistical test cannot rescue a poor study design or low quality data.
